# Shakespeare: agent weights vs random initialisation

Trains the character-level LSTM from wsp-initializer's Shakespeare experiment
from several starting points, each with exactly the same recipe, and compares
how they learn.

1. Runtime → Change runtime type → GPU.
2. Run all. When asked, pick the agent's `final.weights.h5` from its run folder.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

# The recipe, as in wsp/experiments/shakespeare.py. Every starting point is trained the same way.
SEQ_LEN = 100
EMBEDDING_DIM, UNITS, DROPOUT = 128, 512, 0.3
LEARNING_RATE, CLIPNORM = 2e-3, 1.0
WARMUP_EPOCHS = 1  # the learning rate rises linearly from 0 to LEARNING_RATE over these
BATCH_SIZE = 64
EPOCHS, PATIENCE = 50, 3

AGENT_WEIGHTS = "final.weights.h5"
SEEDS = [0, 1, 2]  # one training run per seed for each starting point

## The agent's weights

Pick the agent's `final.weights.h5` from its run folder.

In [ ]:
from google.colab import files

uploaded = files.upload()

# Save it as AGENT_WEIGHTS whatever it is called: Colab renames a repeat upload
# (e.g. "final.weights (1).h5"), and Keras only loads files ending in .weights.h5.
name, data = next(iter(uploaded.items()))
with open(AGENT_WEIGHTS, "wb") as f:
    f.write(data)

print(f"Saved {name} as {AGENT_WEIGHTS}")

## Data

The same split as the experiment: the first 90% of the text for training (what
the agent saw), the next 5% for validation (early stopping here, and the
agent's `evaluate`), and the last 5% for test, which is only scored.

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
text = open(tf.keras.utils.get_file("shakespeare.txt", URL), encoding="utf-8").read()
vocab = np.array(sorted(set(text)))
ids = np.searchsorted(vocab, list(text)).astype(np.int32)


def passages(ids):
    """
    Cut ids into chunks of SEQ_LEN + 1: x is all but the last character, y all but the first.
    """
    count = len(ids) // (SEQ_LEN + 1)
    chunks = ids[: count * (SEQ_LEN + 1)].reshape(count, SEQ_LEN + 1)
    return chunks[:, :-1], chunks[:, 1:]


a, b = int(len(ids) * 0.90), int(len(ids) * 0.95)
train, validation, test = passages(ids[:a]), passages(ids[a:b]), passages(ids[b:])
print(f"{len(vocab)} characters; {len(train[0]):,} training passages")


def make_dataset(split, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(split)
    if shuffle:
        ds = ds.shuffle(10_000)
    return ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

## Model

The same network as `build_model` in the experiment. The layer names must match
for the agent's weights to load.

In [ ]:
def build_model():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(None,), dtype="int32"),
        tf.keras.layers.Embedding(len(vocab), EMBEDDING_DIM, name="embedding"),
        tf.keras.layers.LSTM(UNITS, return_sequences=True, name="lstm_layer_1"),
        tf.keras.layers.Dropout(DROPOUT),
        tf.keras.layers.LSTM(UNITS, return_sequences=True, name="lstm_layer_2"),
        tf.keras.layers.Dropout(DROPOUT),
        tf.keras.layers.Dense(len(vocab), name="output"),
    ])

## Starting points

Each returns a fresh model. "random + unigram bias" is a cheap data-aware
baseline: random weights, but the output bias set to the log frequency of each
character. The agent should beat it, not just plain random weights.

In [ ]:
def random_start():
    return build_model()  # Keras's usual random initialisation


def unigram_start():
    model = build_model()
    counts = np.bincount(train[1].ravel(), minlength=len(vocab)) + 1
    model.get_layer("output").bias.assign(np.log(counts / counts.sum()))
    return model


def agent_start():
    model = build_model()
    model.load_weights(AGENT_WEIGHTS)
    return model


STARTS = {
    "random": random_start,
    "random + unigram bias": unigram_start,
    "agent": agent_start,
}

## Training

The learning rate warms up linearly from 0 over the first epoch, then stays at
`LEARNING_RATE`. The seed sets the random weights, the shuffling and the dropout. Losses are
converted from nats to bits per character.

In [ ]:
BITS = 1 / np.log(2)


def train_from(start, seed):
    tf.keras.utils.set_random_seed(seed)
    model = start()
    steps_per_epoch = len(train[0]) // BATCH_SIZE
    learning_rate = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0,
        warmup_target=LEARNING_RATE,  # rise linearly from 0 to this...
        warmup_steps=WARMUP_EPOCHS * steps_per_epoch,  # ...over the warm-up
        decay_steps=1,
        alpha=1.0,  # then stay there: no decay
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate, clipnorm=CLIPNORM),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )

    before, _ = model.evaluate(make_dataset(validation), verbose=0)
    history = model.fit(
        make_dataset(train, shuffle=True),
        validation_data=make_dataset(validation),
        epochs=EPOCHS,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)],
        verbose=2,
    )
    test_loss, test_accuracy = model.evaluate(make_dataset(test), verbose=0)

    return {
        "curve": [loss * BITS for loss in [before, *history.history["val_loss"]]],
        "test_bits": test_loss * BITS,
        "test_accuracy": test_accuracy,
    }

In [ ]:
results = []

for name, start in STARTS.items():
    for seed in SEEDS:
        print(f"\n=== {name}, seed {seed} ===")
        started = time.time()
        run = train_from(start, seed)
        results.append({"start": name, "seed": seed, "minutes": (time.time() - started) / 60, **run})

## Results

Validation bits per character after every epoch. Epoch 0 is the starting point,
before any training.

In [ ]:
colours = dict(zip(STARTS, ["tab:grey", "tab:blue", "tab:red"]))

for r in results:
    label = r["start"] if r["seed"] == SEEDS[0] else None
    plt.plot(r["curve"], color=colours[r["start"]], alpha=0.7, label=label)

plt.xlabel("epoch (0 = before training)")
plt.ylabel("validation bits per character")
plt.ylim(top=4)  # random weights start at about 6 bits, which would flatten the rest
plt.legend()
plt.show()

In [ ]:
table = pd.DataFrame([
    {
        "start": r["start"],
        "seed": r["seed"],
        "bits at start": r["curve"][0],
        "bits after 1 epoch": r["curve"][1],
        "best epoch": int(np.argmin(r["curve"])),
        "best bits": min(r["curve"]),
        "test bits": r["test_bits"],
        "test accuracy": r["test_accuracy"],
        "minutes": r["minutes"],
    }
    for r in results
])

display(table.round(3))
table.drop(columns="seed").groupby("start", sort=False).mean().round(3)